# Walidacja LPPLS względem 14 historycznych okresów baniek BTC (2015–2024)

Notatnik tworzy **(1) wykres ceny BTC z okresami baniek i alarmami LPPLS**, **(2) wykres LPPLS Confidence Indicator** oraz **(3) tabelę oceny 14 epizodów**. Wykorzystuje zapisane dopasowania — **nie uruchamia ponownej estymacji LPPLS**.

**Interpretacja:** etykiety `UNION01`–`UNION14` oznaczają historyczne okresy opisane w literaturze. Porównanie z nimi jest walidacją retrospektywną, nie dowodzi prognozowania krachów. `confidence` oznacza odsetek zaakceptowanych okien, **nie** skalibrowane prawdopodobieństwo bańki.

W folderze notatnika umieść `lppls_rolling_results_z_etykietami_baniek.csv`. Jeżeli masz cenę BTC z tego samego eksperymentu, umieść również `btc_usd_2015_2024.csv` z kolumnami `Date` i `price`. W przeciwnym razie notatnik spróbuje pobrać ceny z Yahoo Finance (wymaga połączenia internetowego).

In [ ]:
import sys as _sys
from pathlib import Path as _ResearchPath
_sys.path.insert(0,str(_ResearchPath('.').resolve()/'src'))
from lppls_research import TemporalAvailabilityValidator
# 0. Biblioteki i katalog wyników
# Jeżeli trzeba, odkomentuj:
# %pip install pandas numpy matplotlib yfinance scikit-learn

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

ROOT = Path('.')
LPPLS_CSV = ROOT / 'lppls_rolling_results_z_etykietami_baniek.csv'
BTC_CSV = ROOT / 'btc_usd_2015_2024.csv'  # preferowany plik cen z generowania LPPLS
OUTPUT = ROOT / '04_wyniki_walidacji_lppls'
OUTPUT.mkdir(parents=True, exist_ok=True)

R2_THRESHOLD = 0.90
PREWARNING_DAYS = 30

assert LPPLS_CSV.exists(), f'Brak pliku: {LPPLS_CSV.resolve()}'


In [ ]:
# 1. Referencyjne epizody baniek z artykułu

data_union = [
    ['UNION01', '2016-06-08', '2016-06-18'],
    ['UNION02', '2016-12-30', '2017-01-15'],
    ['UNION03', '2017-02-22', '2017-03-07'],
    ['UNION04', '2017-04-27', '2017-07-05'],
    ['UNION05', '2017-07-15', '2018-01-09'],
    ['UNION06', '2019-06-17', '2019-07-13'],
    ['UNION07', '2019-08-09', '2019-08-10'],
    ['UNION08', '2019-12-06', '2020-01-05'],
    ['UNION09', '2020-02-07', '2020-02-17'],
    ['UNION10', '2020-03-20', '2020-09-21'],
    ['UNION11', '2020-11-06', '2021-05-20'],
    ['UNION12', '2021-08-08', '2021-09-19'],
    ['UNION13', '2021-10-05', '2021-12-02'],
    ['UNION14', '2024-03-04', '2024-03-25'],
]
episodes = pd.DataFrame(data_union, columns=['bubble_id', 'start', 'end'])
episodes['start'] = pd.to_datetime(episodes['start'])
episodes['end'] = pd.to_datetime(episodes['end'])
assert (episodes['start'] <= episodes['end']).all()
assert len(episodes) == 14


In [ ]:
# 2. Wczytanie dopasowań, filtracja i agregacja do poziomu dnia
fits = pd.read_csv(LPPLS_CSV, parse_dates=['win_start', 'win_end'])
needed = {'win_start','win_end','ok','cond_m','cond_w','cond_B','cond_tc','r2','literature_bubble'}
assert needed.issubset(fits.columns), f'Brak kolumn: {sorted(needed - set(fits.columns))}'

# Use a shared acceptance rule and daily aggregation for all notebooks.
from lppls_research import build_lppls_daily_features, LPPLSAcceptanceCriteria
criteria=LPPLSAcceptanceCriteria(r2_min=R2_THRESHOLD)
evaluated=criteria.evaluate(fits)
calendar=pd.date_range(evaluated.win_end.min(),evaluated.win_end.max(),freq='D')
features=build_lppls_daily_features(fits,calendar=calendar,criteria=criteria)
episode_status=(fits.groupby('win_end')['literature_bubble'].max()
                .rename('literature_bubble').rename_axis('date').reset_index())
features=features.merge(episode_status,on='date',how='left',validate='one_to_one')
features['literature_bubble']=features.literature_bubble.fillna(0).astype(int)
daily=features.rename(columns={'lppls_n_all':'n_all',
        'lppls_n_accepted':'n_accepted','lppls_ci':'confidence'})[
        ['date','n_all','n_accepted','literature_bubble','confidence']].copy()
daily['lppls_alarm'] = daily['n_accepted'].gt(0).astype(int)

# Zabezpieczenie: ponownie odtwórz etykiety z 14 epizodów i sprawdź ich zgodność z plikiem
daily['reference_check'] = 0
for row in episodes.itertuples(index=False):
    mask = daily['date'].between(row.start, row.end)
    daily.loc[mask, 'reference_check'] = 1
assert daily['reference_check'].equals(daily['literature_bubble']), 'Etykiety w CSV nie zgadzają się z listą 14 epizodów!'
daily = daily.drop(columns='reference_check')

assert daily['date'].is_unique
assert daily['date'].diff().dropna().eq(pd.Timedelta(days=1)).all(), 'Dni w wynikach nie są ciągłe — sprawdź brakujące daty'

print('Wszystkich okien:', len(fits))
print('Zaakceptowanych:', int(evaluated.accepted.sum()))
print('Dni z co najmniej jednym alarmem:', int(daily['lppls_alarm'].sum()))
print('Zakres dostępnych sygnałów:', daily['date'].min().date(), '–', daily['date'].max().date())
daily.to_csv(OUTPUT / 'lppls_daily_validation.csv', index=False)
display(daily.head())


## Wykres 1 — cena BTC, epizody baniek i alarmy LPPLS

Dla wiernego odtworzenia eksperymentu **najlepiej użyć dokładnie tych samych cen**, które posłużyły do estymacji LPPLS. Jeśli plik cen nie istnieje, poniższa komórka pobierze serię BTC-USD z Yahoo Finance i zapisze ją do CSV; należy sprawdzić, czy odpowiada serii bazowej eksperymentu.

Dla czytelności zastosowano skalę logarytmiczną ceny (można usunąć `ax.set_yscale('log')`).

In [ ]:
# 3. Wczytanie serii cen BTC (lub pobranie, jeśli lokalnego pliku brak)
if BTC_CSV.exists():
    price_df = pd.read_csv(BTC_CSV)
    print('Używam lokalnego pliku cen:', BTC_CSV)
else:
    raise FileNotFoundError(f'Missing frozen BTC closing prices: {BTC_CSV}')

# Dopuszczamy warianty kolumn spotykane w notatnikach
if 'price' not in price_df.columns:
    if 'Close' in price_df.columns: price_df = price_df.rename(columns={'Close':'price'})
    elif 'Adj Close' in price_df.columns: price_df = price_df.rename(columns={'Adj Close':'price'})
    else: raise ValueError('Plik cen musi zawierać kolumnę price, Close lub Adj Close')
if 'Date' not in price_df.columns:
    if 'date' in price_df.columns: price_df = price_df.rename(columns={'date':'Date'})
    else: raise ValueError('Brak kolumny Date w pliku cen')

price_df['Date'] = pd.to_datetime(price_df['Date'], utc=True).dt.tz_convert(None).dt.normalize()
price_df['price'] = pd.to_numeric(price_df['price'], errors='coerce')
price_df = (price_df.dropna(subset=['Date','price'])
           .sort_values('Date').drop_duplicates('Date').reset_index(drop=True))
assert (price_df['price'] > 0).all()
assert price_df['Date'].min() <= daily['date'].min(), 'Ceny rozpoczynają się zbyt późno'
assert price_df['Date'].max() >= daily['date'].max(), 'Ceny kończą się zbyt wcześnie'

# Ceny dzienne w punktach alarmowych
alarm_days = daily.loc[daily['lppls_alarm'].eq(1), ['date','confidence']].merge(
    price_df[['Date','price']], left_on='date', right_on='Date', how='left', validate='one_to_one')
assert alarm_days['price'].notna().all(), 'Dla części alarmów brakuje cen BTC'


In [ ]:
# 4. Wykres ceny BTC, baniek z literatury i alarmów LPPLS
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(price_df['Date'], price_df['price'], linewidth=1.15, color='#262626', label='BTC-USD (Close)')

for row in episodes.itertuples(index=False):
    ax.axvspan(row.start, row.end, color='#d95050', alpha=0.18, zorder=0)

ax.scatter(alarm_days['date'], alarm_days['price'], s=17, color='#007f8c',
           alpha=0.78, marker='o', linewidths=0, zorder=3)
ax.set_yscale('log')
ax.set_title('Bitcoin: historyczne epizody baniek i sygnały LPPLS (2015–2024)')
ax.set_xlabel('Data')
ax.set_ylabel('Cena BTC-USD [USD] (skala logarytmiczna)')
ax.grid(alpha=.18)
ax.legend(handles=[
    Line2D([0],[0],color='#262626',lw=1.5,label='Cena BTC'),
    Patch(facecolor='#d95050',alpha=.25,label='Epizod bańki z literatury'),
    Line2D([0],[0],linestyle='None',marker='o',color='#007f8c',markersize=5,label='Dzień z alarmem LPPLS')
],loc='upper left')
fig.tight_layout()
fig.savefig(OUTPUT / '01_cena_btc_banki_alarmy_lppls.png', dpi=220, bbox_inches='tight')
plt.show()


## Wykres 2 — LPPLS Confidence Indicator

\(CI_t = N_{accepted,t}/N_{all,t}\). Jest to odsetek zaakceptowanych dopasowań kończących się w dniu \(t\). Wartość 0 oznacza brak dopasowań spełniających kryteria; nie oznacza jednak udowodnionego braku bańki.

In [ ]:
# 5. Wykres Confidence Indicator
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(daily['date'], daily['confidence'], color='#087e81', linewidth=0.85,
        label='LPPLS Confidence Indicator')
for row in episodes.itertuples(index=False):
    ax.axvspan(row.start, row.end, color='#d95050', alpha=.17, zorder=0)
ax.set_ylim(0, max(0.05, float(daily['confidence'].max()) * 1.06))
ax.set_xlabel('Data')
ax.set_ylabel('Odsetek zaakceptowanych okien')
ax.set_title('LPPLS Confidence Indicator na tle historycznych baniek')
ax.grid(alpha=.20)
ax.legend(handles=[
    Line2D([0],[0],color='#087e81',lw=1.5,label='Confidence Indicator'),
    Patch(facecolor='#d95050',alpha=.25,label='Epizod bańki')
],loc='upper left')
fig.tight_layout()
fig.savefig(OUTPUT / '02_lppls_confidence_indicator.png', dpi=220, bbox_inches='tight')
plt.show()


## Tabela 3 — ocena każdego z 14 epizodów

**Wykryty epizod** oznacza co najmniej **jeden alarm w trakcie** jego trwania. To bardzo łagodne kryterium; do interpretacji trzeba również podać udział dni z alarmem. **Wcześniejsze ostrzeżenie** to alarm w ciągu 30 dni przed początkiem referencyjnej bańki, poza innymi epizodami referencyjnymi. To nie jest tożsame z ostrzeżeniem przed krachem.

In [ ]:
# 6. Szczegółowa ewaluacja 14 epizodów
records = []
for ep in episodes.itertuples(index=False):
    during = daily.loc[daily['date'].between(ep.start, ep.end)].copy()
    # Dni poprzedzające epizod, ale bez okresów należących do wcześniejszych baniek
    pre = daily.loc[
        (daily['date'] >= ep.start - pd.Timedelta(days=PREWARNING_DAYS)) &
        (daily['date'] < ep.start) &
        (daily['literature_bubble'] == 0)
    ].copy()

    alarms_during = during.loc[during['lppls_alarm'].eq(1), 'date']
    alarms_pre = pre.loc[pre['lppls_alarm'].eq(1), 'date']
    first_during = alarms_during.min() if not alarms_during.empty else pd.NaT
    closest_pre = alarms_pre.max() if not alarms_pre.empty else pd.NaT

    records.append({
        'bubble_id': ep.bubble_id,
        'start': ep.start.date().isoformat(),
        'end': ep.end.date().isoformat(),
        'days_in_episode': len(during),
        'days_with_alarm': int(during['lppls_alarm'].sum()),
        'alarm_day_share': round(float(during['lppls_alarm'].mean()) if len(during) else np.nan, 3),
        'detected': bool(len(alarms_during) > 0),
        'first_alarm_in_episode': first_during.date().isoformat() if pd.notna(first_during) else '',
        'prewarning_30d': bool(len(alarms_pre) > 0),
        'alarms_in_30d_before': int(len(alarms_pre)),
        'closest_prewarning_date': closest_pre.date().isoformat() if pd.notna(closest_pre) else '',
        'days_before_start': int((ep.start - closest_pre).days) if pd.notna(closest_pre) else np.nan,
    })

result = pd.DataFrame(records)
result.to_csv(OUTPUT / '03_ocena_14_epizodow_lppls.csv', index=False)

display(result)
print('Epizody z alarmem w trakcie:', int(result['detected'].sum()), '/', len(result))
print('Epizody z wcześniejszym sygnałem do 30 dni:', int(result['prewarning_30d'].sum()), '/', len(result))


In [ ]:
# 7. Uzupełniające metryki dzienne — pomocne do artykułu
truth = daily['literature_bubble']
pred = daily['lppls_alarm']
print('Macierz pomyłek (TN, FP / FN, TP):')
print(confusion_matrix(truth, pred))
print('Precision:', round(precision_score(truth, pred, zero_division=0), 3))
print('Recall:', round(recall_score(truth, pred, zero_division=0), 3))
print('F1:', round(f1_score(truth, pred, zero_division=0), 3))
print('ROC-AUC (Confidence Indicator):', round(roc_auc_score(truth, daily['confidence']), 3))
print('Average Precision (Confidence Indicator):', round(average_precision_score(truth, daily['confidence']), 3))
print('Pliki wynikowe:', OUTPUT.resolve())


### Jak interpretować wyniki

- Wykres 1 pokazuje, **kiedy** pojawiał się alarm na tle ceny i referencyjnych epizodów. Nie dowodzi, że alarm przewidywał przyszły spadek.
- Wykres 2 pokazuje **siłę zgodności dopasowań** kończących się w danym dniu, bez uczenia modelu na etykietach baniek.
- Tabela pokazuje **wykrywanie epizodów i pokrycie ich dni alarmami**. Dni poza epizodami referencyjnymi nie muszą oznaczać pewnego braku bańki.
- Wyniki są **in-sample w sensie doboru kryteriów**: progi filtracji nie zostały wybrane w niezależnym teście. Jest to audyt/retrospektywna walidacja LPPLS, nie walidacja walk-forward modelu predykcyjnego.
- Etykiety literaturowe częściowo korzystają z LPPLS; dodatkowa walidacja zewnętrzna powinna analizować osobno epizody SADF/GSADF i niezależnie definiowane spadki cen.
